# 01 – Data Cleaning

Merged from `Data_Cleaning.ipynb` and the cleaning portion of `Cleaning_Hannah.ipynb`.

This notebook covers:
1. Loading all raw SVH files
2. Null-value audit across every file
3. Department name/city cleanup
4. Diagnosis-family share checks (used to justify our 3 diagnosis family choices)
5. County code -> County name -> Region mapping
6. Merging patients with Census (TIGER) block group data for geographic analysis

**Note:** wait-time computation and cancer/fracture/depression analysis has been moved to `05_wait_time_analysis.ipynb` since that's analysis, not cleaning.

**Confidentiality:** raw CSVs are gitignored and never committed. Do not add `.head()`/print statements on raw patient rows without clearing outputs before committing.

## 1. Load Raw Data

In [ ]:
import pandas as pd
import numpy as np

departments = pd.read_csv("departments.csv")
diagnosis = pd.read_csv("diagnosis.csv")
encounters = pd.read_csv("encounters.csv")
patients = pd.read_csv("patients.csv")
providers = pd.read_csv("providers.csv")
social_determinants = pd.read_csv("social_determinants.csv")
tigercensuscodes = pd.read_csv("tigercensuscodes.csv")
supporting = pd.read_csv("Social Determinant Questions and Domains.csv")

## 2. Null Value Audit

Checking missingness across every file before doing any merges or feature engineering.

In [ ]:
print("departments:", departments.shape[0], "rows")
departments.isnull().sum()

In [ ]:
print("diagnosis:", diagnosis.shape[0], "rows")
diagnosis.isnull().sum()

In [ ]:
print("encounters:", encounters.shape[0], "rows")
encounters.isnull().sum()

In [ ]:
print("patients:", patients.shape[0], "rows")
patients.isnull().sum()

In [ ]:
print("providers:", providers.shape[0], "rows")
providers.isnull().sum()

In [ ]:
social_determinants_supported = pd.merge(social_determinants, supporting)
print("social_determinants (merged w/ supporting):", social_determinants_supported.shape[0], "rows")
social_determinants_supported.isnull().sum()

In [ ]:
print("tigercensuscodes:", tigercensuscodes.shape[0], "rows")
tigercensuscodes.isnull().sum()

## 3. Department Name Cleanup

City names have inconsistent casing/spelling for the same city (e.g. `TOPEKA` vs `Topek`, `OSAGE CITY` vs `Osage City`, `MANHATTAN` vs `Manhattan`). Standardize before using `City` for any grouping.

In [ ]:
departments['City'].unique()

# Known duplicates to reconcile:
# TOPEKA / Topek -> Topeka
# OSAGE CITY / Osage City -> Osage City
# MANHATTAN / Manhattan -> Manhattan

In [ ]:
# Standardize city name casing
departments['City'] = departments['City'].str.strip().str.title()

# Fix known non-standard spelling
departments['City'] = departments['City'].replace({'Topek': 'Topeka'})

departments['City'].unique()

In [ ]:
departments['DepartmentType'].unique()

In [ ]:
departments['DepartmentSpecialty'].unique()

# Note: *Deleted, *Not Applicable, *Unspecified all start with an asterisk,
# making them easy to filter out downstream if needed:
# departments[~departments['DepartmentSpecialty'].str.startswith('*', na=False)]

## 4. Diagnosis Family Share Checks

Quick sanity checks on how much of the overall encounter volume each diagnosis family represents. This is what justified choosing **cancer, fractures, and depression** as our 3 core diagnosis families (see `02_diagnosis_family_selection.ipynb` for the full ICD-based grouping logic).

In [ ]:
encounters_diagnosis = pd.merge(encounters, diagnosis, left_on='PrimaryDiagnosisKey', right_on='DiagnosisKey')
encounters_diagnosis['GroupName'].value_counts().head(10)

In [ ]:
encounters_diagnosis['FRACTURE'] = encounters_diagnosis['GroupName'].str.contains('fracture', case=False, na=False).astype(int)
encounters_diagnosis['CANCER'] = encounters_diagnosis['GroupName'].str.contains('malignant', case=False, na=False).astype(int)
encounters_diagnosis['DEPRESSION'] = encounters_diagnosis['GroupName'].str.contains('depressive', case=False, na=False).astype(int)

most_common_share = (encounters_diagnosis['GroupName'] == 
    'Encounter for general examination without complaint, suspected or reported diagnosis').mean() * 100

print(f"Fracture share:        {encounters_diagnosis['FRACTURE'].mean() * 100:.2f}%")
print(f"Cancer share:          {encounters_diagnosis['CANCER'].mean() * 100:.2f}%")
print(f"Depression share:      {encounters_diagnosis['DEPRESSION'].mean() * 100:.2f}%")
print(f"Most common diagnosis: {most_common_share:.2f}%  (for comparison)")

These percentages look small in isolation, but are meaningful relative to the single most common diagnosis category overall (~7.6%) — supporting our choice of these three families as substantial, distinct patient populations.

## 5. County Code -> County Name -> Region Mapping

Builds the lookup tables needed to map patients to their Kansas county and regional planning district, used throughout the geographic/access analyses.

In [ ]:
county_mapping = pd.DataFrame({
    "CountyCode": [
        "001","003","005","007","009","011","013","015","017","019","021","023","025","027",
        "029","031","033","035","037","039","041","043","045","047","049","051","053","055",
        "057","059","061","063","065","067","069","071","073","075","077","079","081","083",
        "085","087","089","091","093","095","097","099","101","103","105","107","109","111",
        "113","115","117","119","121","123","125","127","129","131","133","135","137","139",
        "141","143","145","147","149","151","153","155","157","159","161","163","165","167",
        "169","171","173","175","177","179","181","183","185","187","189","191","193","195",
        "197","199","201","203","205","207","209"
    ],
    "CountyName": [
        "Allen","Anderson","Atchison","Barber","Barton","Bourbon","Brown","Butler","Chase",
        "Chautauqua","Cherokee","Cheyenne","Clark","Clay","Cloud","Coffey","Comanche","Cowley",
        "Crawford","Decatur","Dickinson","Doniphan","Douglas","Edwards","Elk","Ellis","Ellsworth",
        "Finney","Ford","Franklin","Geary","Gove","Graham","Grant","Gray","Greeley","Greenwood",
        "Hamilton","Harper","Harvey","Haskell","Hodgeman","Jackson","Jefferson","Jewell",
        "Johnson","Kearny","Kingman","Kiowa","Labette","Lane","Leavenworth","Lincoln","Linn",
        "Logan","Lyon","McPherson","Marion","Marshall","Meade","Miami","Mitchell","Montgomery",
        "Morris","Morton","Nemaha","Neosho","Ness","Norton","Osage","Osborne","Ottawa","Pawnee",
        "Phillips","Pottawatomie","Pratt","Rawlins","Reno","Republic","Rice","Riley","Rooks",
        "Rush","Russell","Saline","Scott","Sedgwick","Seward","Shawnee","Sheridan","Sherman",
        "Smith","Stafford","Stanton","Stevens","Sumner","Thomas","Trego","Wabaunsee","Wallace",
        "Washington","Wichita","Wilson","Woodson","Wyandotte"
    ]
})

In [ ]:
region_map = {
    "Shawnee": "Northeast Kansas", "Jefferson": "Northeast Kansas", "Riley": "Northeast Kansas",
    "Lyon": "Northeast Kansas", "Douglas": "Northeast Kansas", "Osage": "Northeast Kansas",
    "Nemaha": "Northeast Kansas", "Wabaunsee": "Northeast Kansas", "Pottawatomie": "Northeast Kansas",
    "Jackson": "Northeast Kansas", "Johnson": "Northeast Kansas", "Franklin": "Northeast Kansas",
    "Coffey": "Northeast Kansas", "Chase": "Northeast Kansas", "Atchison": "Northeast Kansas",
    "Geary": "Northeast Kansas", "Wyandotte": "Northeast Kansas", "Marshall": "Northeast Kansas",
    "Morris": "Northeast Kansas", "Leavenworth": "Northeast Kansas", "Brown": "Northeast Kansas",
    "Doniphan": "Northeast Kansas", "Miami": "Northeast Kansas",

    "Cheyenne": "Northwest Kansas", "Rawlins": "Northwest Kansas", "Decatur": "Northwest Kansas",
    "Norton": "Northwest Kansas", "Phillips": "Northwest Kansas", "Sherman": "Northwest Kansas",
    "Thomas": "Northwest Kansas", "Sheridan": "Northwest Kansas", "Graham": "Northwest Kansas",
    "Rooks": "Northwest Kansas", "Wallace": "Northwest Kansas", "Logan": "Northwest Kansas",
    "Gove": "Northwest Kansas", "Trego": "Northwest Kansas", "Ellis": "Northwest Kansas",

    "Smith": "North Central Kansas", "Jewell": "North Central Kansas", "Republic": "North Central Kansas",
    "Washington": "North Central Kansas", "Osborne": "North Central Kansas", "Mitchell": "North Central Kansas",
    "Cloud": "North Central Kansas", "Clay": "North Central Kansas", "Lincoln": "North Central Kansas",
    "Ottawa": "North Central Kansas", "Russell": "North Central Kansas", "Saline": "North Central Kansas",
    "Ellsworth": "North Central Kansas", "Dickinson": "North Central Kansas",

    "Ford": "Southwest Kansas", "Finney": "Southwest Kansas", "Scott": "Southwest Kansas",
    "Grant": "Southwest Kansas", "Pratt": "Southwest Kansas", "Edwards": "Southwest Kansas",
    "Barber": "Southwest Kansas", "Ness": "Southwest Kansas",

    "Sedgwick": "South Central Kansas", "McPherson": "South Central Kansas", "Harvey": "South Central Kansas",
    "Cowley": "South Central Kansas", "Butler": "South Central Kansas", "Pawnee": "South Central Kansas",
    "Reno": "South Central Kansas", "Rush": "South Central Kansas", "Barton": "South Central Kansas",
    "Rice": "South Central Kansas", "Harper": "South Central Kansas", "Stafford": "South Central Kansas",

    "Woodson": "Southeast Kansas", "Greenwood": "Southeast Kansas", "Neosho": "Southeast Kansas",
    "Allen": "Southeast Kansas", "Wilson": "Southeast Kansas", "Labette": "Southeast Kansas",
    "Montgomery": "Southeast Kansas", "Crawford": "Southeast Kansas", "Chautauqua": "Southeast Kansas",
    "Bourbon": "Southeast Kansas", "Elk": "Southeast Kansas", "Linn": "Southeast Kansas",
    "Anderson": "Southeast Kansas"
}

region_df = pd.DataFrame(region_map.items(), columns=["County", "Region"])
region_df = pd.merge(region_df, county_mapping, left_on="County", right_on="CountyName").drop(columns="CountyName")
region_df.head()

## 6. Merge Patients with Census (TIGER) Data

Filters out patients with missing/unspecified census block info, then merges patient-level data with TIGER census block group population data to enable geographic analysis.

In [ ]:
# Removing patients with null or unspecified census block group information.
# NOTE: flagged in original notes as an open question — revisit if this drops a
# disproportionate share of any one region/demographic.
patients_clean = patients[
    patients["CensusBlockGroupFipsCode"].notna() &
    (patients["CensusBlockGroupFipsCode"] != "*Unspecified")
].copy()

tigercensuscodes["GEOID"] = tigercensuscodes["GEOID"].astype(str)
tigercensuscodes["CountyFips"] = tigercensuscodes["GEOID"].str[:5]

tiger_geo = tigercensuscodes[["GEOID", "CountyFips", "CENTLAT", "CENTLON", "PopulationValue"]]

patients_geo = patients_clean.merge(
    tiger_geo,
    left_on="CensusBlockGroupFipsCode",
    right_on="GEOID",
    how="left"
)

patients_geo["CountyCode"] = patients_geo["CountyFips"].str[-3:]
patients_geo.head()

## 7. Export Cleaned Data for Downstream Notebooks

Save cleaned/derived tables so `02_diagnosis_family_selection.ipynb`, `03_geographic_access_map.ipynb`, `04_journey_clustering.ipynb`, and `05_wait_time_analysis.ipynb` can load them directly instead of re-running this cleaning pipeline.

**Reminder:** these exports still contain patient-level data — keep them in the gitignored data folder, not the repo.

In [ ]:
# Adjust the output path to wherever your local (gitignored) data folder lives
region_df.to_csv("../data/derived/region_df.csv", index=False)
patients_geo.to_csv("../data/derived/patients_geo.csv", index=False)
encounters_diagnosis.to_csv("../data/derived/encounters_diagnosis.csv", index=False)

print("Cleaned tables exported to ../data/derived/")